In [ ]:
import pandas as pd
file_path = ('/content/drive/MyDrive/HALE-Data/merge.csv')
data = pd.read_csv(file_path, encoding='ISO-8859-1')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print("Shape:", data.shape)
missing_values = data.isnull().sum()
print("Missing values:")
print(missing_values)

Shape: (12210, 16)
Missing values:
Period                                                                                                                                                        0
ParentLocation                                                                                                                                                0
Location                                                                                                                                                      0
Dim1                                                                                                                                                          0
HALE_Birth                                                                                                                                                    0
HALE_60                                                                                                                                                       0
infan

In [ ]:
data_filtered = data[(data['Period'] >= 2000) & (data['Period'] <= 2019)]  #for the period 2000-2019

threshold = 0.6 * len(data_filtered)
data_filtered_cleaned = data_filtered.dropna(thresh=threshold, axis=1)

In [ ]:
missing_values = data_filtered_cleaned.isnull().sum()
print("Missing values after filter:")
print(missing_values)

Missing values after filter:
Period                                                                                                                                                       0
ParentLocation                                                                                                                                               0
Location                                                                                                                                                     0
Dim1                                                                                                                                                         0
HALE_Birth                                                                                                                                                   0
HALE_60                                                                                                                                                      0
infant mortality 

In [ ]:
for country in data_filtered_cleaned['Location'].unique():
    country_data = data_filtered_cleaned[data_filtered_cleaned['Location'] == country]
    country_2000_data = country_data[country_data['Period'] == 2000]

    if country_2000_data.isnull().values.any():
        future_data = country_data[country_data['Period'] > 2000].sort_values('Period')


        if not future_data.empty:
            data_filtered_cleaned.loc[(data_filtered_cleaned['Location'] == country) &
                                      (data_filtered_cleaned['Period'] == 2000)] = (
                data_filtered_cleaned.loc[(data_filtered_cleaned['Location'] == country) &
                                          (data_filtered_cleaned['Period'] == 2000)].fillna(future_data.iloc[0])
            )

data_filtered_cleaned = data_filtered_cleaned.round(2)

In [ ]:
missing_values = data_filtered_cleaned.isnull().sum()
print("Missing values after backward filling:")
print(missing_values)

Missing values after backward filling:
Period                                                                                                                                                       0
ParentLocation                                                                                                                                               0
Location                                                                                                                                                     0
Dim1                                                                                                                                                         0
HALE_Birth                                                                                                                                                   0
HALE_60                                                                                                                                                      0
infant 

In [ ]:
data_interpolated = data_filtered_cleaned.groupby('Location').apply(lambda group: group.interpolate(method='linear'))
data_interpolated = data_interpolated.round(2)


/tmp/ipykernel_3875/28790754.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  data_interpolated = data_filtered_cleaned.groupby('Location').apply(lambda group: group.interpolate(method='linear'))
/tmp/ipykernel_3875/28790754.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  data_interpolated = data_filtered_cleaned.groupby('Location').apply(lambda group: group.interpolate(method='linear'))
/tmp/ipykernel_3875/28790754.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  data_interpolated = data_filtered_cleaned.groupby('Location').apply(lambda group: group.interpolate(method='linear'))
/tmp/ipykernel_3875/28790

In [ ]:
missing_values = data_interpolated.isnull().sum()
print("Missing values after interpolation:")
print(missing_values)

Missing values after interpolation:
Period                                                                                                                                                       0
ParentLocation                                                                                                                                               0
Location                                                                                                                                                     0
Dim1                                                                                                                                                         0
HALE_Birth                                                                                                                                                   0
HALE_60                                                                                                                                                      0
infant mor

In [ ]:
data_filled = data_interpolated.fillna(data_interpolated.mean(numeric_only=True))
data_filled_rounded = data_filled.round(2)

In [ ]:
missing_values = data_filled_rounded.isnull().sum()
print("Missing values after global mean imputation:")
print(missing_values)

Missing values after global mean imputation:
Period                                                                                                                                                     0
ParentLocation                                                                                                                                             0
Location                                                                                                                                                   0
Dim1                                                                                                                                                       0
HALE_Birth                                                                                                                                                 0
HALE_60                                                                                                                                                    0
infant mortal

In [ ]:
import os
directory_path = '/data/raw'
if not os.path.exists(directory_path):
    os.makedirs(directory_path)
    print(f"Directory '{directory_path}' created.")

data_filled_rounded.to_csv(os.path.join(directory_path, 'cleaned_dataset.csv'), index=False)

Directory '/data/raw' created.
